In [1]:
from pathlib import Path
import pandas as pd

DATA = Path('../data')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## A

In [2]:
# veo qué encoding usa A
for enc in ('utf-8', 'latin1'):
    df = pd.read_csv(DATA / 'listings_plataforma_A.csv', encoding=enc)
    print(enc, df['antiguedad'].dropna().head(2).tolist(), df['cochera'].value_counts(dropna=False).to_dict())

utf-8 ['4 años', '6'] {'No': 2285, 'Sí': 1377}
latin1 ['4 aÃ±os', '6'] {'No': 2285, 'SÃ\xad': 1377}


In [3]:
# cargo A y miro nulos por columna
a = pd.read_csv(DATA / 'listings_plataforma_A.csv', encoding='latin1')
print(a.shape)
(a.isna().mean() * 100).round(1).sort_values(ascending=False)

(3662, 18)


amenities              90.8
expensas               45.7
superficie_cubierta    13.9
dormitorios             9.9
antiguedad              8.7
banios                  7.6
titulo                  0.0
id_aviso                0.0
precio                  0.0
zona_barrio             0.0
ambientes               0.0
superficie_total        0.0
tipo                    0.0
cochera                 0.0
balcon                  0.0
latitud                 0.0
longitud                0.0
fecha_publicacion       0.0
dtype: float64

In [4]:
# cuántas filas hay en cada formato de precio
p = a['precio'].astype(str)
pd.Series({
    'USD':       p.str.contains(r'USD|U\$S|us\$', case=False, regex=True).sum(),
    'pesos $':   p.str.contains(r'^\$', regex=True).sum(),
    'sin signo': p.str.match(r'^[\d.]+$').sum(),
    'NaN-like':  p.isna().sum() + p.str.lower().isin(['', 'nan', 'consultar']).sum(),
})

USD           349
pesos $       784
sin signo    2483
NaN-like       46
dtype: int64

In [5]:
# qué rango de precio sale en cada formato
def num(s):
    return pd.to_numeric(s.str.extract(r'([\d.]+)')[0].str.replace('.', '', regex=False), errors='coerce')

for label, mask in [('USD',   p.str.contains(r'USD|U\$S', case=False, regex=True)),
                    ('$',     p.str.contains(r'^\$', regex=True)),
                    ('plain', p.str.match(r'^[\d.]+$'))]:
    print(label, num(p[mask]).describe().round(0).to_dict())

USD {'count': 349.0, 'mean': 912.0, 'std': 592.0, 'min': 136.0, '25%': 502.0, '50%': 735.0, '75%': 1155.0, 'max': 3846.0}
$ {'count': 784.0, 'mean': 98235180.0, 'std': 276847458.0, 'min': 1.0, '25%': 1000.0, '50%': 740000.0, '75%': 1293750.0, 'max': 999999999.0}
plain {'count': 2483.0, 'mean': 997408.0, 'std': 676296.0, 'min': 150000.0, '25%': 549500.0, '50%': 807000.0, '75%': 1235500.0, 'max': 7712000.0}


In [6]:
# cuántas vienen con sufijos sucios o coma decimal
print('lat con coma:', a['latitud'].astype(str).str.contains(',').sum())
print('lng con coma:', a['longitud'].astype(str).str.contains(',').sum())
print('superficie con "m":', a['superficie_total'].astype(str).str.contains('m', case=False).sum())
print('ambientes con "amb":', a['ambientes'].astype(str).str.contains('amb', case=False).sum())
print()
print(a['antiguedad'].dropna().astype(str).pipe(lambda s: s[~s.str.match(r'^\d+$')]).value_counts().head(5))

lat con coma: 298
lng con coma: 286
superficie con "m": 2203
ambientes con "amb": 1476

antiguedad
A estrenar    54
3 aÃ±os       33
2 aÃ±os       32
6 aÃ±os       31
5 aÃ±os       30
Name: count, dtype: int64


In [7]:
# expensas: rango y cola alta
pd.to_numeric(a['expensas'], errors='coerce').describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0)

count        1990.0
mean       619057.0
std       2748585.0
min          7775.0
50%         86686.0
90%        246942.0
95%       2720835.0
99%      12177922.0
max      45204700.0
Name: expensas, dtype: float64

In [8]:
# cuántos barrios crudos vs después de strip+lower
zb = a['zona_barrio'].astype(str)
print(zb.nunique(), '->', zb.str.strip().str.lower().nunique())

50 -> 15


## B

In [9]:
# cargo B, busco columnas constantes y nulos
b = pd.read_csv(DATA / 'listings_plataforma_B.csv', encoding='utf-8')
print(b.shape)
print('constantes:', [c for c in b.columns if b[c].nunique(dropna=True) <= 1])
(b.isna().mean() * 100).round(1).sort_values(ascending=False)

(3639, 18)
constantes: ['category_l1', 'category_l2']


age_years          29.4
total_area_m2      19.2
price_ars           5.9
covered_area_m2     5.1
bathrooms           3.9
rooms               3.8
bedrooms            3.2
listing_id          0.0
category_l1         0.0
neighborhood        0.0
property_type       0.0
category_l2         0.0
has_garage          0.0
has_balcony         0.0
has_pool            0.0
lat                 0.0
lng                 0.0
created_at          0.0
dtype: float64

In [10]:
# rango del precio en B
b['price_ars'].describe(percentiles=[0.5, 0.99]).round(0)

count       3426.0
mean      989860.0
std       670412.0
min       166000.0
50%       809000.0
99%      3613250.0
max      7712000.0
Name: price_ars, dtype: float64

## C

In [11]:
# cargo C y miro nulos
c = pd.read_csv(DATA / 'listings_plataforma_C.csv', encoding='utf-8')
print(c.shape, list(c.columns))
(c.isna().mean() * 100).round(1).sort_values(ascending=False)

(3699, 9) ['codigo', 'operacion', 'tipo_inmueble', 'barrio', 'precio_publicado', 'superficie', 'descripcion', 'coordenadas', 'publicado']


coordenadas         39.6
superficie           5.0
descripcion          0.5
codigo               0.0
operacion            0.0
precio_publicado     0.0
barrio               0.0
tipo_inmueble        0.0
publicado            0.0
dtype: float64

In [12]:
# cuántos precios traen "/mes", "Consultar" o USD
pp = c['precio_publicado'].astype(str)
print('/mes:', pp.str.contains('/mes').sum(),
      '| Consultar:', pp.str.contains('Consultar', case=False).sum(),
      '| USD:', pp.str.contains(r'USD|u\$s', case=False, regex=True).sum())

/mes: 3405 | Consultar: 294 | USD: 0


In [13]:
# qué saco de la descripción con regex
desc = c['descripcion'].fillna('')
for pat, label in [(r'\d+\s*amb', 'amb'), (r'\d+\s*dorm', 'dorm'),
                   (r'\d+\s*ba', 'baños'), (r'\d+\s*a[ñn]', 'años'),
                   ('cochera', 'cochera'), (r'pileta|piscina', 'pileta'),
                   (r'balc[oó]n', 'balcon')]:
    print(f'{label}: {desc.str.contains(pat, case=False, regex=True).sum()}')

amb: 2954
dorm: 2627
baños: 2237
años: 1613
cochera: 980
pileta: 362
balcon: 1202


In [14]:
# cuántas filas traen coordenadas
co = c['coordenadas'].fillna('')
print('no-vacias:', co.ne('').sum(), '/', len(c))

no-vacias: 2235 / 3699


## cruce

In [15]:
# qué barrios comparten las 3 plataformas
norm = lambda s: s.astype(str).str.strip().str.lower()
ba, bb, bc = set(norm(a['zona_barrio'])), set(norm(b['neighborhood'])), set(norm(c['barrio']))
print(f'A={len(ba)} B={len(bb)} C={len(bc)} ABC={len(ba & bb & bc)} union={len(ba | bb | bc)}')
print('solo A:', sorted(ba - bb - bc))

A=15 B=12 C=12 ABC=11 union=16
solo A: ['nunez', 'nuã\x91ez', 'nuã±ez', 'v. crespo']


In [16]:
# tipos de propiedad por plataforma
for n, col, df in [('A', 'tipo', a), ('B', 'property_type', b), ('C', 'tipo_inmueble', c)]:
    print(n, df[col].str.lower().value_counts().to_dict())

A {'departamento': 2831, 'ph': 547, 'casa': 284}
B {'departamento': 2857, 'ph': 532, 'casa': 250}
C {'departamento': 2916, 'ph': 520, 'casa': 263}


In [17]:
# rango de fechas en cada plataforma
fa = pd.to_datetime(a['fecha_publicacion'], format='%d/%m/%Y', errors='coerce')
fb = pd.to_datetime(b['created_at'], errors='coerce')
fc = pd.to_datetime(c['publicado'], format='%d-%m-%Y', errors='coerce')
for n, s in [('A', fa), ('B', fb), ('C', fc)]:
    print(n, s.min().date(), s.max().date(), 'nan:', s.isna().sum())

A 2024-01-01 2025-08-23 nan: 0
B 2024-01-01 2025-08-23 nan: 0
C 2024-01-01 2025-08-23 nan: 0
